# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the dataset using the `mlcroissant` library.

### Dataset Source
[FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) ([schema JSON-LD](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json))

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset basic info
print(f"Name: {metadata.name}")
print(f"Description:\n{metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")
print(f"Personal Sensitive Information: {getattr(metadata, 'personalSensitiveInformation', [])}")


## 2. Data Overview
Review available record sets, including their `@id` and fields.

In [ ]:
# List available record sets and their fields using the Croissant metadata
print("Available record sets and their field @ids:")
record_sets = []
for record_set in metadata.record_sets:
    print(f"- Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id}) type: {getattr(field, 'data_type', 'Unknown')}")
    else:
        print("  No fields found.")
    print()
    record_sets.append(record_set.id)
if not record_sets:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from each available record set into a DataFrame. Use record set and field `@id`s as shown in the overview above.

If there are no record sets, the data might instead be available under the `distribution` property or as part of the dataset as a whole. Below, we attempt to load records from all available record sets.

In [ ]:
# Extract data from each record set and show columns (by @id)
dataframes = {}

for record_set_id in record_sets:
    # Use records generator
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded record set {record_set_id} with columns:")
            print(list(df.columns))
            display(df.head())
        else:
            print(f"\nNo records found for record set {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

if not dataframes:
    print("No tabular dataframes were loaded. Please check the record sets in the metadata or explore raw distributions.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

<br>
*If you know the relevant numeric and group fields by their `@id`, set them below. Otherwise, see the record sets/fields in the previous cell.*

In [ ]:
# Example setup -- Define a record set and field @ids to use for EDA
from IPython.display import display

# Pick the first available record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Try to automatically detect numeric fields (float or int)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for EDA. Please review columns above.")
    else:
        print(f"Using numeric field: {numeric_field_id} for filtering and normalization.")

        # Set a threshold (mean for demonstration)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows")
        display(filtered_df.head())

        # Add normalized column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (z-score):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to choose a group field
        candidate_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            print(f"Grouping filtered data by {group_field} and showing mean of numeric fields:")
            group_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            display(group_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No dataframes available for EDA. Please review earlier steps.")

## 5. Visualization
Visualize data distributions or relationships between fields. If using a notebook, interactive plots can be rendered inline. Below is an example plot of a numeric field distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals() and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
    
    # If grouping field present, boxplot
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, you loaded, previewed, and explored a dataset using the `mlcroissant` package, referencing all record sets and fields by their `@id` as per Croissant best practices. Summary of findings and next analysis steps can go here.